# 📊 Feature Analysis & Selection

**Purpose:** Analyze computed features, run feature selection, and visualize importance rankings.

---

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.core.config import load_config
from src.core.database import DatabaseManager
from src.features.store import FeatureStore
from src.features.selection import FeatureSelector
from src.features.registry import feature_registry

plt.style.use('dark_background')

config = load_config()
db = DatabaseManager(config.database.url)
db.initialize()

store = FeatureStore(config, db)
print(f'✅ Feature Registry: {feature_registry.count} features')

## 1. Feature Store Summary

In [ ]:
summary = store.get_feature_summary()
print(f'📋 Feature Groups:')
for group, count in summary.get('features_by_group', {}).items():
    print(f'   {group}: {count} features')

## 2. Compute & Retrieve Features

In [ ]:
symbol = 'BTC/USDT'

# Compute (or retrieve cached) features
features = store.compute_and_store(symbol, '1d')
print(f'📊 {symbol}: {features.shape[1]} features × {features.shape[0]} samples')
features.describe()

## 3. Feature Selection

In [ ]:
selector = FeatureSelector(config, db)

# Run selection with available methods
# (SHAP/XGBoost requires xgboost to be installed)
rankings = selector.run(
    symbol=symbol,
    timeframe='1d',
    methods=['mutual_info', 'permutation', 'pca'],
    top_k=30,
)

if not rankings.empty:
    print('\n🏆 Top 20 Selected Features:')
    print(rankings.head(20).to_string(index=False))

## 4. Visualize Feature Importance

In [ ]:
if not rankings.empty:
    top_20 = rankings.head(20).sort_values('consensus_score')
    
    fig, ax = plt.subplots(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0.3, 0.9, len(top_20)))
    ax.barh(top_20['feature_name'], top_20['consensus_score'], color=colors)
    ax.set_xlabel('Consensus Score (RRF)')
    ax.set_title(f'{symbol} — Top 20 Features by Consensus Ranking', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ No rankings available. Ensure features are computed first.')

## 5. Feature Distribution Analysis

In [ ]:
if not rankings.empty:
    top_6 = rankings['feature_name'].head(6).tolist()
    available = [f for f in top_6 if f in features.columns]
    
    if available:
        fig, axes = plt.subplots(2, 3, figsize=(14, 8))
        for ax, feat_name in zip(axes.flat, available):
            features[feat_name].dropna().hist(bins=50, ax=ax, color='#7b68ee', alpha=0.7)
            ax.set_title(feat_name, fontsize=10)
        plt.suptitle('Distribution of Top Selected Features', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()